# 01 — From a linear score to a probability

*Notebook 1 of 6 — Logistic Regression*

Welcome to logistic regression. We begin with the one idea the whole method rests on: turning a
number — a *score* — into a *probability*.

**Prerequisites.** Module 00: features and the feature space (NB 02); scores and thresholds
(NB 06–08). Chapter 02: a probability is a number between 0 and 1.

**What you'll be able to do.**
- Explain what the sigmoid (logistic) function does, and why we need it.
- Convert freely between a probability, its odds, and its log-odds.
- Turn a feature into a class probability with a linear score and the sigmoid.
- Read the decision point where the probability crosses one half.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ml_course import colors, datasets, viz

viz.use_course_style()
np.random.seed(0)  # no randomness in this notebook, but we keep the habit

# The course fil-rouge: Adelie vs Gentoo penguins. One feature for now: bill length (mm).
df = datasets.load_penguins()
print(df["species"].value_counts())
df.head()

## Where we are

Two methods behind us, each with its own way of deciding:

- **k-nearest neighbours** voted by **distance** — who are my closest neighbours?
- **Naive Bayes** multiplied **probabilities** — how likely is each class, feature by feature?

Logistic regression starts a different idea: draw **one weighted line** through the feature space
and read a probability straight off it. But a line gives a *score*, and a score can be any number at
all, while a probability must sit between 0 and 1. So our first task is to turn a score into a
probability.

(Re-establishing the ground: a **probability** is a number in [0, 1]. So far we have only ever
*counted* them — relative frequencies. Here we will *compute* one from a score.)

## A score is unbounded; a probability is not

A linear score is the familiar weighted sum

$$ z = w \cdot x + b $$

with a weight $w$ for the feature, plus an offset $b$. As $x$ ranges over its values, $z$ can land
**anywhere** on the real line — from large negative to large positive. A probability, though, is
trapped in $[0, 1]$.

We need a function that gently **squashes** the whole real line into $(0, 1)$: large negative scores
near 0, large positive scores near 1, and a smooth ramp in between. That function is the **sigmoid**
(or **logistic**) function, written with the natural exponential $e \approx 2.718$:

$$ \sigma(z) = \frac{1}{1 + e^{-z}}. $$

In [ ]:
def sigmoid(z):
    """Map any real score z to a probability in (0, 1)."""
    return 1.0 / (1.0 + np.exp(-z))


z = np.linspace(-6, 6, 200)

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.plot(z, sigmoid(z), color=colors.COLORS["model"], linewidth=2)
ax.axhline(0.5, color=colors.COLORS["grid"], linewidth=1)
ax.scatter([0], [0.5], color=colors.COLORS["highlight"], zorder=5, s=60)
ax.set_xlabel("score   z = w·x + b")
ax.set_ylabel("sigmoid(z)  =  probability")
ax.set_title("The sigmoid squashes any score into (0, 1)")
plt.show()

**Read the figure.** The S-shaped curve does exactly what we asked: a very negative score is
pushed near **0**, a very positive score near **1**, and the score **z = 0** maps to exactly **one
half** (the highlighted point). It is smooth and always increasing — every score has one
probability, and bigger scores mean bigger probabilities. This single function is how logistic
regression will speak in probabilities.

## Odds, and log-odds

A probability is bounded in $[0, 1]$, which makes it awkward to add or scale. Two re-expressions
fix that.

- The **odds** of an event are $\dfrac{p}{1 - p}$ — "how many times more likely than not". A
  probability of $0.75$ is odds of $3$ (three to one); odds run from $0$ to $\infty$.
- The **log-odds** (or **logit**) are $\log\!\dfrac{p}{1 - p}$ — the log of the odds (the
  **natural** log, base $e$ — the same $e$ as in $\sigma$, which is why the two undo each other).
  These run from $-\infty$ to $+\infty$, exactly the range of a score.

And here is the mirror of the sigmoid: if $p = \sigma(z)$, then inverting gives

$$ z = \log\frac{p}{1 - p}. $$

So **the score *is* the log-odds** of the positive class. The sigmoid turns a score (log-odds) into
a probability; the logit turns a probability back into a score.

In [ ]:
p = np.array([0.10, 0.25, 0.50, 0.75, 0.90])
table = pd.DataFrame(
    {
        "probability p": p,
        "odds  p/(1-p)": p / (1 - p),
        "log-odds  log[p/(1-p)]": np.log(p / (1 - p)),
    }
).round(3)
table

**Read the table.** Odds restate "9 in 10" as "9 to 1". The log-odds are **symmetric around
0**: a probability of exactly one half is log-odds **0**; anything more likely than not is
**positive**; anything less is **negative**. Notice the mirror — p = 0.10 and p = 0.90 give log-odds
of −2.197 and +2.197. The sigmoid and the logit are inverses, carrying us back and forth between a
score and a probability.

## From a feature to a probability

Let's use a real measurement. Among Adélie and Gentoo penguins, Gentoo tend to have **longer
bills**. So a longer `bill_length_mm` should push the probability of "Gentoo" up.

We build the score $z = w \cdot \text{bill\_length} + b$, then read $P(\text{Gentoo}) = \sigma(z)$.
For now we **choose** $w$ and $b$ by hand — only enough to see the machinery turn. Finding the
*best* weights is what NB 3 and NB 4 are about; here we meet the sigmoid on real data.

In [ ]:
# Hand-chosen weights (NOT fitted): a gentle slope with the half-way point at 43 mm.
# NB 3-4 will *find* the best weights; here we set them to watch the mechanism.
w, b = 1.0, -43.0

bill = df["bill_length_mm"].to_numpy()
is_gentoo = (df["species"] == "Gentoo").to_numpy().astype(int)

grid = np.linspace(bill.min(), bill.max(), 200)
p_grid = sigmoid(w * grid + b)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(grid, p_grid, color=colors.COLORS["model"], linewidth=2, label="P(Gentoo) = sigmoid(z)")
for cls, name in [(0, "Adelie"), (1, "Gentoo")]:
    m = is_gentoo == cls
    ax.scatter(
        bill[m],
        is_gentoo[m],
        color=colors.CLASS_CYCLE[cls],
        alpha=0.5,
        edgecolor=colors.COLORS["text"],
        linewidth=0.4,
        s=35,
        label=name,
    )
ax.axhline(0.5, color=colors.COLORS["grid"], linewidth=1)
ax.axvline(
    -b / w,
    color=colors.COLORS["highlight"],
    linestyle="--",
    linewidth=1.5,
    label=f"half at {-b / w:.0f} mm",
)
ax.set_xlabel("bill_length_mm")
ax.set_ylabel("P(Gentoo)")
ax.set_title("A score through the sigmoid turns bill length into P(Gentoo)")
ax.legend(loc="center right")
plt.show()

**Read the figure.** Short bills sit at the bottom-left, where the curve reads
P(Gentoo) ≈ 0 — these are Adélie. Long bills sit top-right, where P ≈ 1 — Gentoo. The curve crosses
**one half at about 43 mm** (the dashed line): that is the bill length where the model is genuinely
undecided. It is **steep through the 41–45 mm overlap band**, where the two species mix, and flat on
either side where they don't. About a fifth of the penguins land where the curve reads between 0.1
and 0.9 — its unsure zone — and there it says, honestly, that it cannot tell them apart.

## Deciding: threshold at one half

A probability is not yet a decision. The natural rule is to call it **Gentoo when $P \ge \tfrac12$**
— equivalently, when the score $z \ge 0$, equivalently when the bill is longer than the half-way
crossing. The one-half cut is a sensible default, not a law: when one kind of mistake costs more
than the other, we move it. NB 6 does exactly that, with care.

In [ ]:
proba = sigmoid(w * bill + b)
pred_gentoo = (proba >= 0.5).astype(int)
accuracy = (pred_gentoo == is_gentoo).mean()
print(f"Hand-chosen rule accuracy on the {len(bill)} penguins: {accuracy:.3f}")

# Three penguins: a clear Adelie, a clear Gentoo, and a borderline one near 43 mm.
borderline = int(np.argmin(np.abs(bill - 43.0)))
short, long = int(np.argmin(bill)), int(np.argmax(bill))
examples = pd.DataFrame(
    {
        "bill_length_mm": bill[[short, borderline, long]],
        "P(Gentoo)": proba[[short, borderline, long]],
        "true species": df["species"].to_numpy()[[short, borderline, long]],
    }
).round(3)
examples

**Read the result.** The hand-chosen line already gets the great majority right — and we have
not fitted anything. The short-billed penguin reads P ≈ 0 (Adélie), the long-billed one P ≈ 1
(Gentoo), and the borderline penguin near 43 mm reads close to one half — the model says "I'm not
sure", which a bare label could never tell you. That honest uncertainty is the gift of working in
probabilities. In NB 3 and NB 4 we will *find* the weights that make this rule as sharp as the data
allows.

## Your turn

1. **Read the curve.** From the sigmoid above, what is $P(\text{Gentoo})$ at a bill length of
   **45 mm**? Compute the score $z = w \cdot 45 + b$ and pass it through `sigmoid`.
2. **Flip the weight.** Set `w = -1.0` (and pick `b` so the crossing stays near 43 mm), then redraw.
   Which way does the curve now lean, and which species does a long bill predict? Explain why.
3. **Solve for the crossing.** The half-point is where $z = 0$, i.e. $\text{bill} = -b/w$. Confirm
   it equals 43 mm. Then, using the log-odds, find the bill length where $P = 0.9$ (hint: there the
   score is $z = \log 9$).

## What you built

- The **sigmoid** $\sigma(z) = 1/(1+e^{-z})$ — the function that turns any score into a probability
  in $(0, 1)$.
- **Odds** and **log-odds**, and the key fact that **a linear score *is* the log-odds** of the
  positive class — the sigmoid and the logit are inverses.
- A first classifier in spirit: a feature → a linear score → the sigmoid → $P(\text{Gentoo})$ → a
  decision at the one-half threshold.

**Vocabulary.** *linear score* $w\cdot x + b$ · *sigmoid / logistic function* · *odds* ·
*log-odds / logit* · *decision threshold*.

You can now turn a number into a probability and read where a decision tips. Next we give that score
a geometry.

## Going further (optional)

What we built is the **logistic model**: $P(y=1 \mid x) = \sigma(w \cdot x + b)$, with the logit as
its **link function**. It belongs to the family of *generalized linear models*, where a linear score
is mapped to a constrained quantity by a chosen link. You do not need this name to continue — but it
is the formal home of the sigmoid.

A look ahead: NB 2 reads the weighted line **geometrically** (the decision boundary, and what each
weight means); NB 3 and NB 4 stop choosing weights by hand and **find** them by minimizing a loss.

## References

- Cox, D. R. (1958). *The regression analysis of binary sequences.* Journal of the Royal Statistical
  Society B, 20(2), 215–242. DOI: 10.1111/j.2517-6161.1958.tb00292.x
- Berkson, J. (1944). *Application of the logistic function to bio-assay.* Journal of the American
  Statistical Association, 39(227), 357–365. DOI: 10.1080/01621459.1944.10500699
- James, G., Witten, D., Hastie, T., & Tibshirani, R. (2021). *An Introduction to Statistical
  Learning* (§4.3). DOI: 10.1007/978-1-0716-1418-1

---

*Previous: Module 02 — Naive Bayes*  ·  *Next: 02 — The decision boundary & reading the weights*